Why we chose to change our dataset for the project?

Initially, we planned to use the NHANES dataset that included information from the "National Health and Nutrition Examination Survey. This dataset was extremely complex with various sub-datasets including information from demographics, lab tests, health exams, dietary data, questionnaire data, etc. It contained various types of data as well. We realized that if we were to use this dataset, we would need to merge multiple sub-datasets together and parse through thousands of columns of different variable types. Also, the data initally comes in XPT files that would need to be merged then converted to CSV. Therefore, we switched to using another dataset found on Kaggle. The dataset is named "Students Performance in Exams", and has a sufficient amount of information and is well-organized. It is a csv file that includes a few variables such as gender, race, parental level of education, school lunch plan, and test preparation level along with numerical data such as exam scores. We felt like this dataset is more straightforward and will allow us to make real hypotheses. This data is also impactful because our project will allow us to understand how varying factors that we may not have initally assumed to affect students' exam performance actually do. 

In [3]:
# Import the necessary libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [4]:
# Load and view the dataset
df = pd.read_csv("../data/StudentsPerformance.csv")
df.head()


,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [ ]:
# Make column names into one string
df.columns = df.columns.str.lower().str.replace(" ", "_")

# Rename some columns with a better description
df = df.rename(columns = {
    "race/ethnicity": "race",
    "lunch": "lunch_plan",
    "test_preparation_course": "test_prep_status"
})
df.head()

,gender,race,parental_level_of_education,lunch_plan,test_prep_status,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [ ]:
# Checking for any missing values. None, so we are good!
df.isnull().sum()

gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

In [ ]:
# Check column data types
df.dtypes

gender                         category
race                           category
parental_level_of_education    category
lunch_plan                     category
test_prep_status               category
math_score                        int64
reading_score                     int64
writing_score                     int64
dtype: object

In [14]:
# The scores are good already since they are integers. We need to make the first five columns into categorical data types, though.
categorical_columns = [
    "gender", "race", "parental_level_of_education", 
    "lunch_plan", "test_prep_status"
]

for col in categorical_columns:
    df[col] = df[col].astype("category")

df.dtypes

gender                         category
race                           category
parental_level_of_education    category
lunch_plan                     category
test_prep_status               category
math_score                        int64
reading_score                     int64
writing_score                     int64
dtype: object

Question #1: Does taking a prep course improve math grades?

Hypothesis test used to interpret conclusion #1: ANOVA on the average math scores between racial groups

Null Hypothesis (Ho): The mean math score is the same across all race/ethnicity groups.
Alternative Hypothesis (Ha): At least one race/ethnicity group has a different mean math score.

In [3]:
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt


groups = [group["math_score"].values for name, group in df.groupby("race")]
f_stat, p_value = f_oneway(*groups)

print("ANOVA (scipy)")
print("F-statistic:", f_stat)
print("p-value:", p_value)


ImportError: cannot import name 'distributions' from 'statsmodels' (/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/__init__.py)

A one-way ANOVA was conducted to compare math scores across racial groups. A Tukey HSD post-hoc test was performed to identify which exact group differed.

In [4]:
tukey = pairwise_tukeyhsd(
    endog=df["math_score"],
    groups=df["race"],
    alpha=0.05
)

print("\nTukey HSD Results:")
print(tukey)

NameError: name 'pairwise_tukeyhsd' is not defined

The initial ANOVA showed a significant influence of race/ethnicity on math scores (F = 14.59 , p-value = 1.37 x 10^-11). Post-hoc analysis shows that there's a statistically significant difference between groups A&D, A&E, B&D, B&E, C&E, and D&E. The other pairings fail to reject the null hypothesis that there is no difference in means amongst math scores.

In [ ]:
plt.figure()
df.boxplot(column="math_score", by="race")
plt.title("Math Score by Race")
plt.xlabel("Race")
plt.ylabel("Math Score")
plt.show()

This box plot suggests that math performance varies significantly across racial/ethnic groups.